<a href="https://colab.research.google.com/github/M7office/Stroke/blob/main/AHA_module_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Slide 2 / Figure 2 — Biological module value differences
# Colab-ready script
#
# Input files expected in the current Colab folder (/content):
#   C_patient_data*.csv
#   C_NPX_data*.csv
#   strokecog_literature_aligned_pathway_framework*.csv
#   strokecog_literature_aligned_protein_pathway_assignment_summary*.csv
#
# Output folder:
#   AHA_slide02_outputs/
# ============================================================

from pathlib import Path
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -----------------------------
# User settings
# -----------------------------
BASE = Path.cwd()  # Colab default: /content
OUT = BASE / "AHA_slide02_outputs_v6"
OUT.mkdir(parents=True, exist_ok=True)

SIS3_CUTOFF = 63
RANDOM_SEED = 42
N_BOOT = 5000

PATHWAY_ASSIGNMENT_MODE = "all"  # "all" or "primary"
FILTER_LOW_DETECTION_PROTEINS = True
LOW_DETECTION_FILTER_MODE = "complete_case"

FIGURE_TITLE = "Biological Module Values by SIS3 Group"

COLORS = {
    "lower": "#D55E00",
    "higher": "#0072B2",
    "effect": "#2E5C68",
    "black": "#303030",
    "gray": "#6E6E6E",
    "lightgray": "#D9D9D9",
    "verylight": "#EAEAEA",
    "stripe": "#F7F6F2",
}

MODULES = [
    "Serotonin / tryptophan-kynurenine / monoamine metabolism",
    "Vesicle secretion / extracellular vesicle / membrane trafficking",
    "Cell death / cellular stress / proteostasis",
    "Metabolic / lipid / atherosclerosis / mitochondrial-energy biology",
    "mTOR / MAPK / NF-kB / growth-survival signaling",
    "Hormone / neuroendocrine / HPA-like systemic signaling",
    "Systemic organ injury / leakage / comorbidity markers",
    "Peripheral immune / inflammatory activation",
    "Complement / coagulation / platelet axis",
    "Endothelial / BBB / neurovascular unit",
    "Synaptic / neuronal plasticity / neurotrophic signaling",
    "Integrin / ECM / cell adhesion / vascular remodeling",
]

MODULE_SHORT_LABELS = {
    MODULES[0]: "Serotonin / kynurenine / monoamine",
    MODULES[1]: "Vesicle / extracellular vesicle trafficking",
    MODULES[2]: "Cell death / stress / proteostasis",
    MODULES[3]: "Metabolic / lipid / mitochondrial-energy",
    MODULES[4]: "mTOR / MAPK / NF-κB signaling",
    MODULES[5]: "Hormone / neuroendocrine signaling",
    MODULES[6]: "Systemic injury / comorbidity markers",
    MODULES[7]: "Peripheral immune / inflammatory activation",
    MODULES[8]: "Complement / coagulation / platelet axis",
    MODULES[9]: "Endothelial / BBB / neurovascular unit",
    MODULES[10]: "Synaptic / neuroplasticity / neurotrophic",
    MODULES[11]: "Integrin / ECM / vascular remodeling",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.2,
    "axes.titlesize": 10.5,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 8.8,
    "legend.fontsize": 8.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 450,
})

# -----------------------------
# Helper functions
# -----------------------------
def read_csv_safely(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            df.columns = df.columns.astype(str).str.strip()
            return df
        except UnicodeDecodeError:
            pass
    df = pd.read_csv(path)
    df.columns = df.columns.astype(str).str.strip()
    return df


def csv_files():
    return sorted(BASE.glob("*.csv"))


def find_csv(contains_all=None, contains_any=None, required=True, label="file", exclude_any=None):
    contains_all = [x.lower() for x in (contains_all or [])]
    contains_any = [x.lower() for x in (contains_any or [])]
    exclude_any = [x.lower() for x in (exclude_any or [])]
    matches = []
    for f in csv_files():
        name = f.name.lower()
        if contains_all and not all(x in name for x in contains_all):
            continue
        if contains_any and not any(x in name for x in contains_any):
            continue
        if exclude_any and any(x in name for x in exclude_any):
            continue
        matches.append(f)
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not find {label}. Tried all={contains_all}, any={contains_any}")
        return None
    return sorted(matches, key=lambda p: (len(p.name), p.name))[0]


def find_col(df, candidates, required=True, label="column", avoid=None):
    avoid = [a.lower() for a in (avoid or [])]
    lower_to_col = {c.lower(): c for c in df.columns}
    for cand in candidates:
        c = lower_to_col.get(cand.lower())
        if c is not None and not any(a in c.lower() for a in avoid):
            return c
    for c in df.columns:
        c_l = c.lower()
        if any(cand.lower() in c_l for cand in candidates):
            if not any(a in c_l for a in avoid):
                return c
    if required:
        raise ValueError(f"Could not find {label}. Tried {candidates}. Available columns: {df.columns.tolist()}")
    return None


def normalize_id_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\.0$", "", s)
    return s


def normalize_oid_value(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def normalize_feature_column_name(c):
    s = str(c).strip()
    m = re.search(r"(OID\d+)", s, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    return s


def unique_preserve(seq):
    out, seen = [], set()
    for x in seq:
        if pd.isna(x):
            continue
        x = str(x).strip()
        if x and x not in seen:
            out.append(x)
            seen.add(x)
    return out


def detect_time_years(series, colname=""):
    x = pd.to_numeric(series, errors="coerce")
    lname = str(colname).lower()
    max_val = np.nanmax(x.values) if np.isfinite(x).any() else np.nan
    if "year" in lname or "yrs" in lname or "yr" in lname:
        return x
    if "day" in lname or (np.isfinite(max_val) and max_val > 40):
        return x / 365.25
    if "month" in lname or (np.isfinite(max_val) and max_val > 6):
        return x / 12.0
    return x


def zscore_df(df):
    numeric = df.apply(pd.to_numeric, errors="coerce")
    return (numeric - numeric.mean(axis=0)) / numeric.std(axis=0, ddof=0).replace(0, np.nan)


def bootstrap_mean_diff_stats(a, b, n_boot=N_BOOT, seed=RANDOM_SEED):
    """Bootstrap inference for mean(a) - mean(b)."""
    a = pd.Series(a).dropna().astype(float).values
    b = pd.Series(b).dropna().astype(float).values
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan, np.nan

    diff = float(np.mean(a) - np.mean(b))
    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boot[i] = np.mean(aa) - np.mean(bb)
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])

    pooled_mean = (np.sum(a) + np.sum(b)) / (len(a) + len(b))
    a_null = a - np.mean(a) + pooled_mean
    b_null = b - np.mean(b) + pooled_mean
    boot_null = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a_null, size=len(a_null), replace=True)
        bb = rng.choice(b_null, size=len(b_null), replace=True)
        boot_null[i] = np.mean(aa) - np.mean(bb)
    p_boot = float(np.mean(np.abs(boot_null) >= abs(diff)))
    return diff, float(ci_low), float(ci_high), p_boot


def fdr_bh(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full_like(p, np.nan)
    valid = np.isfinite(p)
    pv = p[valid]
    m = len(pv)
    if m == 0:
        return q
    order = np.argsort(pv)
    ranked = pv[order]
    raw_q = ranked * m / np.arange(1, m + 1)
    adj_q = np.minimum.accumulate(raw_q[::-1])[::-1]
    adj_q = np.minimum(adj_q, 1.0)
    out = np.empty_like(pv)
    out[order] = adj_q
    q[valid] = out
    return q


def fmt_2(x):
    if pd.isna(x):
        return "NA"
    return f"{float(x):.2f}"


def fmt_mean_sd(mean, sd):
    if pd.isna(mean) or pd.isna(sd):
        return "NA"
    return f"{mean:.2f} ({sd:.2f})"


def fmt_ci(diff, lo, hi):
    if pd.isna(diff) or pd.isna(lo) or pd.isna(hi):
        return "NA"
    return f"{diff:.2f} ({lo:.2f} to {hi:.2f})"


def wrap_caption(text, width=160):
    return "\n".join(textwrap.wrap(text, width=width, break_long_words=False))


def symmetric_axis_limit(values, min_limit=0.4, step=0.1, pad=0.04):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return min_limit
    lim = np.max(np.abs(vals)) + pad
    lim = max(min_limit, lim)
    lim = np.ceil(lim / step) * step
    return float(lim)

# -----------------------------
# 1. Load raw data and calculate module scores
# -----------------------------
print("Current folder:", BASE)
print("CSV files found:")
for f in csv_files():
    print(" -", f.name)

patient_file = find_csv(contains_any=["patient"], required=True, label="patient file")
npx_file = find_csv(contains_any=["npx"], required=True, label="NPX file")
framework_file = find_csv(contains_all=["pathway_framework"], required=True, label="pathway framework file")
assignment_file = find_csv(contains_all=["protein_pathway_assignment"], required=True, label="protein pathway assignment file")

patient = read_csv_safely(patient_file)
npx = read_csv_safely(npx_file)
framework = read_csv_safely(framework_file)
assign = read_csv_safely(assignment_file)

pathway_col_fw = find_col(framework, ["literature_aligned_pathway", "pathway"], label="framework pathway column")
framework_order = unique_preserve(framework[pathway_col_fw])
# Keep only the 12 modules, in the intended biological order.
framework_order = [m for m in MODULES if m in framework_order]
if len(framework_order) != 12:
    print(f"Warning: expected 12 modules; found {len(framework_order)} matching modules.")

patient_id_col = find_col(
    patient,
    ["patient_id", "patientid", "participant_id", "subject_id", "sample_id", "record_id", "pid", "id"],
    required=False,
    label="patient ID column",
    avoid=["sis", "score", "time", "age"],
)
if patient_id_col is None:
    patient_id_col = patient.columns[0]

sis3_col = find_col(patient, ["sis3", "sis_3", "sis 3"], label="SIS3 column")
time_col = find_col(patient, ["time_year", "time_years", "years", "month", "day", "time_since", "timesince", "time"], label="time column")

patient = patient.copy()
patient["_patient_id_norm"] = patient[patient_id_col].map(normalize_id_value)
patient["sis3"] = pd.to_numeric(patient[sis3_col], errors="coerce")
patient["time_years"] = detect_time_years(patient[time_col], time_col)

npx = npx.copy()
ref_ids = set(patient["_patient_id_norm"].dropna().astype(str))
id_col_npx = None
best_overlap = -1
for c in npx.columns:
    vals = set(npx[c].map(normalize_id_value).dropna().astype(str))
    overlap = len(vals.intersection(ref_ids))
    if overlap > best_overlap:
        best_overlap = overlap
        id_col_npx = c

npx_value_col = find_col(npx, ["npx", "value"], required=False, label="NPX value column")
oid_col_npx = find_col(npx, ["oid", "olinkid", "assay", "protein_id"], required=False, label="OID column in NPX file")

if best_overlap > 0 and npx_value_col is not None and oid_col_npx is not None and id_col_npx != oid_col_npx:
    npx["_patient_id_norm"] = npx[id_col_npx].map(normalize_id_value)
    npx["_OID_norm"] = npx[oid_col_npx].map(normalize_oid_value)
    npx_wide = npx.pivot_table(index="_patient_id_norm", columns="_OID_norm", values=npx_value_col, aggfunc="mean").reset_index()
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
elif best_overlap > 0:
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    id_col_norm = normalize_feature_column_name(id_col_npx)
    npx_wide["_patient_id_norm"] = npx_wide[id_col_norm].map(normalize_id_value)
    merged = patient.merge(npx_wide, on="_patient_id_norm", how="inner")
else:
    if len(npx) != len(patient):
        raise ValueError("Could not match patient IDs, and NPX row count does not match patient row count.")
    npx_wide = npx.rename(columns={c: normalize_feature_column_name(c) for c in npx.columns})
    npx_wide["_row_order"] = np.arange(len(npx_wide))
    patient["_row_order"] = np.arange(len(patient))
    merged = patient.merge(npx_wide, on="_row_order", how="inner")
    merged["_patient_id_norm"] = merged["_patient_id_norm"].fillna(merged["_row_order"].astype(str))
    print("\nNote: NPX file has no matching patient ID column. Using row-order alignment because row counts match.")

if merged.empty:
    raise ValueError("Patient and NPX files did not merge.")

oid_col_assign = find_col(assign, ["OID", "oid", "olinkid", "protein_id"], label="OID column in assignment file")
if PATHWAY_ASSIGNMENT_MODE == "primary":
    pathway_col_assign = find_col(assign, ["primary_literature_aligned_pathway_draft", "primary_literature_aligned_pathway", "pathway"], label="primary pathway column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
elif PATHWAY_ASSIGNMENT_MODE == "all":
    pathway_col_assign = find_col(assign, ["all_literature_aligned_pathways_draft", "all_literature_aligned_pathways", "all_pathways", "pathways"], label="all-pathways column")
    mapping = assign[[oid_col_assign, pathway_col_assign]].copy()
    mapping.columns = ["OID", "pathway"]
    mapping["pathway"] = mapping["pathway"].astype(str).str.split(r"\s*[;|,]\s*", regex=True)
    mapping = mapping.explode("pathway")
else:
    raise ValueError("PATHWAY_ASSIGNMENT_MODE must be 'primary' or 'all'.")

mapping["OID"] = mapping["OID"].map(normalize_oid_value)
mapping["pathway"] = mapping["pathway"].astype(str).str.strip()
mapping = mapping.dropna()
mapping = mapping[mapping["pathway"].isin(framework_order)]

protein_cols = [c for c in merged.columns if re.fullmatch(r"OID\d+", str(c), flags=re.IGNORECASE)]
protein_cols = [normalize_feature_column_name(c) for c in protein_cols]
protein_cols = [c for c in protein_cols if c in merged.columns]
protein_numeric = merged[protein_cols].apply(pd.to_numeric, errors="coerce")

if FILTER_LOW_DETECTION_PROTEINS and LOW_DETECTION_FILTER_MODE == "complete_case":
    detection_rate = protein_numeric.notna().sum(axis=0) / len(protein_numeric)
    retained_protein_cols = detection_rate[detection_rate == 1.0].index.tolist()
    print(f"\nProteins before complete-case filter: {len(protein_cols)}")
    print(f"Proteins retained after complete-case filter: {len(retained_protein_cols)}")
    protein_cols = retained_protein_cols
    protein_numeric = protein_numeric[protein_cols]
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()
else:
    mapping = mapping[mapping["OID"].isin(protein_cols)].copy()

if mapping.empty:
    raise ValueError("No pathway-assignment OIDs matched retained NPX protein columns.")

protein_z = zscore_df(protein_numeric)
score_df = merged[["_patient_id_norm", "sis3", "time_years"]].copy()
module_membership = []
for module in framework_order:
    oids = sorted(set(mapping.loc[mapping["pathway"] == module, "OID"]).intersection(protein_z.columns))
    score_df[module] = protein_z[oids].mean(axis=1) if len(oids) else np.nan
    module_membership.append({"module": module, "n_proteins": len(oids), "OIDs": ";".join(oids)})

score_df["sis3_level"] = np.where(score_df["sis3"] <= SIS3_CUTOFF, "Lower SIS3 (≤63)", "Higher SIS3 (>63)")
score_df.to_csv(OUT / "slide02_patient_level_module_values.csv", index=False)
pd.DataFrame(module_membership).to_csv(OUT / "slide02_module_membership.csv", index=False)
mapping.to_csv(OUT / "slide02_protein_pathway_mapping_used.csv", index=False)

# -----------------------------
# 2. Summary statistics
# -----------------------------
rows = []
for i, module in enumerate(framework_order):
    low = pd.to_numeric(score_df.loc[score_df["sis3_level"] == "Lower SIS3 (≤63)", module], errors="coerce").dropna()
    high = pd.to_numeric(score_df.loc[score_df["sis3_level"] == "Higher SIS3 (>63)", module], errors="coerce").dropna()
    diff, lo, hi, p = bootstrap_mean_diff_stats(low, high, seed=RANDOM_SEED + i)
    rows.append({
        "No.": f"{i + 1:02d}",
        "Biological module": module,
        "Biological module, short": MODULE_SHORT_LABELS.get(module, module),
        "Lower SIS3, n": len(low),
        "Higher SIS3, n": len(high),
        "Lower SIS3, mean": float(low.mean()),
        "Lower SIS3, SD": float(low.std(ddof=1)),
        "Higher SIS3, mean": float(high.mean()),
        "Higher SIS3, SD": float(high.std(ddof=1)),
        "Mean difference, Lower - Higher": diff,
        "95% CI lower": lo,
        "95% CI upper": hi,
        "P value": p,
    })

stats = pd.DataFrame(rows)
stats["FDR q"] = fdr_bh(stats["P value"].values)
stats["Lower SIS3, mean (SD)"] = [fmt_mean_sd(m, s) for m, s in zip(stats["Lower SIS3, mean"], stats["Lower SIS3, SD"])]
stats["Higher SIS3, mean (SD)"] = [fmt_mean_sd(m, s) for m, s in zip(stats["Higher SIS3, mean"], stats["Higher SIS3, SD"])]
stats["Mean difference (95% CI)"] = [fmt_ci(d, l, h) for d, l, h in zip(stats["Mean difference, Lower - Higher"], stats["95% CI lower"], stats["95% CI upper"])]
stats["P value formatted"] = stats["P value"].map(fmt_2)
stats["FDR q formatted"] = stats["FDR q"].map(fmt_2)
stats.to_csv(OUT / "slide02_module_low_vs_high_statistics.csv", index=False)

# -----------------------------
# 3A. Journal-style bar plot version
# -----------------------------
y = np.arange(len(framework_order))
bar_h = 0.34
low_means = stats["Lower SIS3, mean"].values
high_means = stats["Higher SIS3, mean"].values
long_labels = [f"{i + 1:02d}. {m}" for i, m in enumerate(framework_order)]

fig_bar = plt.figure(figsize=(12.2, 8.6))
gs_bar = GridSpec(3, 1, figure=fig_bar, height_ratios=[0.42, 5.0, 1.55], hspace=0.05)

title_ax = fig_bar.add_subplot(gs_bar[0, 0])
title_ax.axis("off")
title_ax.text(0.00, 0.72, "Figure 2. Biological Module Values by SIS3 Group", ha="left", va="center", fontsize=13.0, fontweight="bold", color=COLORS["black"], transform=title_ax.transAxes)

ax = fig_bar.add_subplot(gs_bar[1, 0])
ax.barh(y - bar_h / 2, low_means, height=bar_h, color=COLORS["lower"], label="Lower SIS3 (≤63)")
ax.barh(y + bar_h / 2, high_means, height=bar_h, color=COLORS["higher"], label="Higher SIS3 (>63)")
ax.axvline(0, color=COLORS["black"], lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels(long_labels)
ax.invert_yaxis()
ax.set_xlabel("Mean standardized biological-module value")
ax.set_xlim(-0.4, 0.4)
ax.set_xticks(np.arange(-0.4, 0.41, 0.1))
ax.set_axisbelow(True)
ax.grid(axis="x", color=COLORS["verylight"], lw=0.7)
ax.tick_params(axis="both", length=3, color=COLORS["gray"])
ax.legend(frameon=False, loc="lower right", handlelength=1.6)

cap_ax = fig_bar.add_subplot(gs_bar[2, 0])
cap_ax.axis("off")
caption_bar = (
    "Lower SIS3 was defined as SIS3 ≤63 and higher SIS3 as SIS3 >63. Biological-module values were calculated as the mean of z-standardized protein NPX "
    "values assigned to each biological module. Positive values indicate higher standardized values, and negative values indicate lower standardized "
    "values relative to the study cohort mean. Orange bars represent lower-SIS3 patients and blue bars represent higher-SIS3 patients."
)
cap_ax.text(0.00, 0.70, wrap_caption(caption_bar, width=150), ha="left", va="top", fontsize=8.6, color=COLORS["black"], linespacing=1.20, transform=cap_ax.transAxes)

fig_bar.subplots_adjust(left=0.44, right=0.97, top=0.93, bottom=0.10)
for ext in ["png", "pdf", "svg"]:
    fig_bar.savefig(OUT / f"figure2_module_values_barplot_journal_style_v6.{ext}", bbox_inches="tight")
plt.close(fig_bar)

# -----------------------------
# 3B. Forest plot version of Table 1
# -----------------------------
N = len(stats)
y = np.arange(N)
fig = plt.figure(figsize=(14.0, 8.0))
gs = GridSpec(
    2, 6,
    figure=fig,
    height_ratios=[0.40, 5.6],
    width_ratios=[0.45, 3.4, 2.55, 1.80, 0.80, 0.80],
    hspace=0.06,
    wspace=0.05,
)

title_ax = fig.add_subplot(gs[0, :])
title_ax.axis("off")
title_ax.text(0.00, 0.74, "Figure 2. " + FIGURE_TITLE, ha="left", va="center", fontsize=13.0, fontweight="bold", color=COLORS["black"], transform=title_ax.transAxes)

ax_no = fig.add_subplot(gs[1, 0])
ax_label = fig.add_subplot(gs[1, 1], sharey=ax_no)
ax_forest = fig.add_subplot(gs[1, 2], sharey=ax_no)
ax_effect = fig.add_subplot(gs[1, 3], sharey=ax_no)
ax_p = fig.add_subplot(gs[1, 4], sharey=ax_no)
ax_q = fig.add_subplot(gs[1, 5], sharey=ax_no)

text_axes = [ax_no, ax_label, ax_effect, ax_p, ax_q]
for a in text_axes:
    a.set_ylim(-0.7, N - 0.5)
    a.invert_yaxis()
    a.axis("off")

ax_forest.set_ylim(-0.7, N - 0.5)
ax_forest.invert_yaxis()

# Alternating row background for table-like readability
for i in range(N):
    if i % 2 == 1:
        for a in [ax_no, ax_label, ax_forest, ax_effect, ax_p, ax_q]:
            a.axhspan(i - 0.5, i + 0.5, color=COLORS["stripe"], zorder=0)

# Headers
header_y = -0.85
ax_no.text(0.0, header_y, "No.", fontsize=9.2, fontweight="bold", ha="left", va="bottom")
ax_label.text(0.0, header_y, "Biological module", fontsize=9.2, fontweight="bold", ha="left", va="bottom")
ax_forest.set_title("Mean difference (95% CI)", loc="left", pad=6, fontsize=9.2, fontweight="bold")
ax_effect.text(0.0, header_y, "Estimate (95% CI)", fontsize=9.2, fontweight="bold", ha="left", va="bottom")
ax_p.text(0.0, header_y, "P value", fontsize=9.2, fontweight="bold", ha="left", va="bottom")
ax_q.text(0.0, header_y, "FDR q", fontsize=9.2, fontweight="bold", ha="left", va="bottom")

# Left and right text columns
for i, row in stats.iterrows():
    ax_no.text(0.0, i, row["No."], ha="left", va="center", fontsize=8.8, color=COLORS["black"])
    ax_label.text(0.0, i, row["Biological module, short"], ha="left", va="center", fontsize=8.8, color=COLORS["black"])
    ax_effect.text(0.0, i, row["Mean difference (95% CI)"], ha="left", va="center", fontsize=8.8, color=COLORS["black"])
    ax_p.text(0.0, i, row["P value formatted"], ha="left", va="center", fontsize=8.8, color=COLORS["black"])
    ax_q.text(0.0, i, row["FDR q formatted"], ha="left", va="center", fontsize=8.8, color=COLORS["black"])

# Forest plot
x = stats["Mean difference, Lower - Higher"].values
lo = stats["95% CI lower"].values
hi = stats["95% CI upper"].values
ax_forest.hlines(y, lo, hi, color=COLORS["effect"], lw=1.35, zorder=3)
ax_forest.plot(x, y, marker="s", linestyle="None", color=COLORS["effect"], markersize=4.8, zorder=4)
ax_forest.axvline(0, color=COLORS["black"], lw=0.8, ls=":", zorder=2)
ax_forest.set_yticks([])
ax_forest.set_xlabel("Mean difference in standardized biological-module value, lower SIS3 − higher SIS3")
forest_lim = symmetric_axis_limit(np.r_[lo, hi, x], min_limit=0.4, step=0.1, pad=0.04)
ax_forest.set_xlim(-forest_lim, forest_lim)
ax_forest.set_axisbelow(True)
ax_forest.grid(axis="x", color=COLORS["verylight"], lw=0.7)
ax_forest.tick_params(axis="x", length=3, color=COLORS["gray"])

caption = (
    "Lower SIS3 was defined as SIS3 ≤63 and higher SIS3 as SIS3 >63. Biological-module values were calculated as the mean of z-standardized protein NPX "
    "values assigned to each biological module. Mean differences are lower SIS3 minus higher SIS3; positive values indicate higher biological-module values "
    "in lower-SIS3 patients. Error bars indicate bootstrap 95% CIs. P values were estimated by bootstrap resampling and FDR q values were calculated "
    "across the 12 biological modules."
)
fig.text(0.02, 0.02, wrap_caption(caption, width=178), ha="left", va="bottom", fontsize=8.6, color=COLORS["black"], linespacing=1.20)
fig.subplots_adjust(left=0.035, right=0.985, top=0.93, bottom=0.15)

for ext in ["png", "pdf", "svg"]:
    fig.savefig(OUT / f"figure2_module_values_forest_journal_style_v6.{ext}", bbox_inches="tight")
plt.close(fig)

print("\nCreated Slide 2 outputs:")
for p in sorted(OUT.glob("figure2_module_values_*.*")):
    print(" -", p)
print("\nModule statistics:")
print(stats[["No.", "Biological module, short", "Lower SIS3, mean (SD)", "Higher SIS3, mean (SD)", "Mean difference (95% CI)", "P value formatted", "FDR q formatted"]].to_string(index=False))


Current folder: /content
CSV files found:
 - C_NPX_data.csv
 - C_patient_data.csv
 - NAME_OID.csv
 - strokecog_literature_aligned_pathway_framework.csv
 - strokecog_literature_aligned_protein_pathway_assignment_summary.csv

Proteins before complete-case filter: 1196
Proteins retained after complete-case filter: 1011

Created Slide 2 outputs:
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_barplot_journal_style_symmetric_v5.pdf
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_barplot_journal_style_symmetric_v5.png
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_barplot_journal_style_symmetric_v5.svg
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_forest_journal_style_symmetric_v5.pdf
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_forest_journal_style_symmetric_v5.png
 - /content/AHA_slide02_outputs_v5/figure2_module_scores_forest_journal_style_symmetric_v5.svg

Module statistics:
No.                    Biological module, short Lower SIS3, mea